## Import dependencies

In [70]:
import pandas as pd
import requests
import json

## Scrape all legislature names and districts to dataframe

In [88]:
# Create blank data frame to store information
legislator_data = []

In [89]:
# Define the years of the General Court stored on the Mass Legislature website
court_numbers = [i for i in range(182, 195)]
court_years =  [i for i in range(2001, 2026, 2)]

# There should be an associated court number for each court year
assert(len(court_years) == len(court_numbers))

In [90]:
# Define global header to be used in all requests
header = {"accept": "application/json"}

In [139]:
# Create a log file for handling parsing errors
f = open('log.txt', 'w')

In [138]:
def scrape_representatives():
    try:

        for i, court_number in enumerate(court_numbers):
            print(f"Parsing data for the {court_number}th General Court...")
            f.write(f"Parsing data for the {court_number}th General Court...")
            url = f"https://malegislature.gov/api/GeneralCourts/{court_number}/LegislativeMembers"
            resp = requests.get(url, headers=header)

            # Make sure response code is correct
            if resp.status_code == 200:
                # Get JSON from response
                print(resp.content)
                data = resp.json()

                # Iterate over each representative returned
                for j, item in enumerate(data):
                    # Output parsing record number
                    print(f"Parsing record [{j} / {len(data)}] for current court")

                    # Get link to xml file
                    xml_url = item["Details"]

                    # Check for broken links containing
                    if '&' in xml_url:
                        f.write(f"Found and skipped invalid url {xml_url} in {court_number}th General Court")
                        continue

                    xml_resp = requests.get(xml_url)

                    f.write(f"[{j}\/{len(data)}] Parsing: {xml_url}\n")

                    # Parse bytes returned into JSON and extract desired fields
                    current_data = []
                    legislator_info = json.loads(xml_resp.content.decode('utf-8'))
                    name = legislator_info['Name']
                    current_data.append(name)
                    current_data.append(legislator_info['Branch'])
                    current_data.append(legislator_info['District'])
                    current_data.append(court_years[i])

                    f.write(f"Found legislator: {name}")

                    legislator_data.append(current_data)

            else:
                print(f"Court Number {court_number} failed to parse!")
                break
        print(f"Finished parsing General Courts {min(court_numbers) - max(court_numbers)}! Found {len(legislator_data)} total records (not unique)")
    except:
        f.write("Error found in parsing")
        f.close()

In [140]:
scrape_representatives()

Parsing data for the 182th General Court...
b'[{"GeneralCourtNumber":182,"MemberCode":"MPS1","Details":"http://malegislature.gov/api/GeneralCourts/182/LegislativeMembers/MPS1"},{"GeneralCourtNumber":182,"MemberCode":"MRK0","Details":"http://malegislature.gov/api/GeneralCourts/182/LegislativeMembers/MRK0"},{"GeneralCourtNumber":182,"MemberCode":"MRP0","Details":"http://malegislature.gov/api/GeneralCourts/182/LegislativeMembers/MRP0"},{"GeneralCourtNumber":182,"MemberCode":"MSR1","Details":"http://malegislature.gov/api/GeneralCourts/182/LegislativeMembers/MSR1"},{"GeneralCourtNumber":182,"MemberCode":"MVF1","Details":"http://malegislature.gov/api/GeneralCourts/182/LegislativeMembers/MVF1"},{"GeneralCourtNumber":182,"MemberCode":"MWM0","Details":"http://malegislature.gov/api/GeneralCourts/182/LegislativeMembers/MWM0"},{"GeneralCourtNumber":182,"MemberCode":"N F1","Details":"http://malegislature.gov/api/GeneralCourts/182/LegislativeMembers/N%20F1"},{"GeneralCourtNumber":182,"MemberCode":"N

In [126]:
# Test calling xml
response = requests.get("http://malegislature.gov/api/GeneralCourts/182/LegislativeMembers/A&F7")

print(response.content)

b'<!DOCTYPE html>\r\n<html>\r\n    <head>\r\n        <title>Runtime Error</title>\r\n        <meta name="viewport" content="width=device-width" />\r\n        <style>\r\n         body {font-family:"Verdana";font-weight:normal;font-size: .7em;color:black;} \r\n         p {font-family:"Verdana";font-weight:normal;color:black;margin-top: -5px}\r\n         b {font-family:"Verdana";font-weight:bold;color:black;margin-top: -5px}\r\n         H1 { font-family:"Verdana";font-weight:normal;font-size:18pt;color:red }\r\n         H2 { font-family:"Verdana";font-weight:normal;font-size:14pt;color:maroon }\r\n         pre {font-family:"Consolas","Lucida Console",Monospace;font-size:11pt;margin:0;padding:0.5em;line-height:14pt}\r\n         .marker {font-weight: bold; color: black;text-decoration: none;}\r\n         .version {color: gray;}\r\n         .error {margin-bottom: 10px;}\r\n         .expandable { text-decoration:underline; font-weight:bold; color:navy; cursor:pointer; }\r\n         @media scr